In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
import librosa
import soundfile as sf

In [2]:
raw_path = Path("data/raw/RAVDESS")
if not raw_path.exists():
    raw_path = Path("../data/raw/RAVDESS")

processed_path = Path("data/processed")
if not processed_path.exists():
    processed_path = Path("../data/processed")

outputs_path = Path("DE_Data_Engineer/outputs")
if not outputs_path.exists():
    outputs_path = Path("../DE_Data_Engineer/outputs")

processed_path.mkdir(parents=True, exist_ok=True)
outputs_path.mkdir(parents=True, exist_ok=True)

In [3]:
files = list(raw_path.rglob("*.wav"))
files = sorted(files)

In [4]:
print("Total WAV files found:", len(files))

Total WAV files found: 1440


In [5]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

records = []
invalid_filenames = []

for f in files:
    parts = f.stem.split("-")
    if len(parts) == 7:
        emotion_code = parts[2]
        actor_id = parts[6]
        emotion = emotion_map.get(emotion_code, "unknown")
        records.append({
            "file_path": str(f),
            "file_name": f.name,
            "actor_id": actor_id,
            "emotion_code": emotion_code,
            "emotion": emotion
        })
    else:
        invalid_filenames.append(f.name)

In [6]:
metadata = pd.DataFrame(records)
print(metadata.head())
print(metadata.shape)

                                           file_path  \
0  ..\data\raw\RAVDESS\Actor_01\03-01-01-01-01-01...   
1  ..\data\raw\RAVDESS\Actor_01\03-01-01-01-01-02...   
2  ..\data\raw\RAVDESS\Actor_01\03-01-01-01-02-01...   
3  ..\data\raw\RAVDESS\Actor_01\03-01-01-01-02-02...   
4  ..\data\raw\RAVDESS\Actor_01\03-01-02-01-01-01...   

                  file_name actor_id emotion_code  emotion  
0  03-01-01-01-01-01-01.wav       01           01  neutral  
1  03-01-01-01-01-02-01.wav       01           01  neutral  
2  03-01-01-01-02-01-01.wav       01           01  neutral  
3  03-01-01-01-02-02-01.wav       01           01  neutral  
4  03-01-02-01-01-01-01.wav       01           02     calm  
(1440, 5)


In [7]:
print(metadata["emotion"].value_counts())

emotion
calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
neutral       96
Name: count, dtype: int64


In [8]:
print(metadata["actor_id"].value_counts().sort_index())

actor_id
01    60
02    60
03    60
04    60
05    60
06    60
07    60
08    60
09    60
10    60
11    60
12    60
13    60
14    60
15    60
16    60
17    60
18    60
19    60
20    60
21    60
22    60
23    60
24    60
Name: count, dtype: int64


In [9]:
print("Missing values in metadata:")
print(metadata.isnull().sum())

Missing values in metadata:
file_path       0
file_name       0
actor_id        0
emotion_code    0
emotion         0
dtype: int64


In [10]:
print("Invalid filenames count:", len(invalid_filenames))
invalid_emotions = metadata[metadata["emotion"] == "unknown"]
print("Invalid emotion codes count:", len(invalid_emotions))

Invalid filenames count: 0
Invalid emotion codes count: 0


In [11]:
sample_rates = []
channels_list = []
durations = []
file_sizes = []
loading_errors = []

for p in metadata["file_path"]:
    try:
        info = sf.info(p)
        sample_rates.append(info.samplerate)
        channels_list.append(info.channels)
        durations.append(info.duration)
        file_sizes.append(Path(p).stat().st_size)
    except Exception as e:
        sample_rates.append(None)
        channels_list.append(None)
        durations.append(None)
        file_sizes.append(None)
        loading_errors.append(str(p))

metadata["sample_rate"] = sample_rates
metadata["channels"] = channels_list
metadata["duration"] = durations
metadata["file_size"] = file_sizes

In [12]:
print(metadata["sample_rate"].value_counts())

sample_rate
16000    1440
Name: count, dtype: int64


In [13]:
print(metadata["channels"].value_counts())

channels
1    1440
Name: count, dtype: int64


In [14]:
print("Min duration:", metadata["duration"].min())
print("Max duration:", metadata["duration"].max())
print("Mean duration:", metadata["duration"].mean())
print("Median duration:", metadata["duration"].median())

Min duration: 2.9363125
Max duration: 5.2719375
Mean duration: 3.7006852430555557
Median duration: 3.670375


In [15]:
print("Audio loading errors:", len(loading_errors))

Audio loading errors: 0


In [16]:
def get_hash(path_str):
    hasher = hashlib.md5()
    with open(path_str, "rb") as f:
        hasher.update(f.read())
    return hasher.hexdigest()

metadata["hash"] = metadata["file_path"].apply(get_hash)

In [17]:
duplicate_mask = metadata.duplicated(subset=["hash"], keep=False)
duplicates = metadata[duplicate_mask].sort_values("hash").copy()
duplicates["duplicate_group"] = duplicates.groupby("hash").ngroup() + 1
print("Duplicate files count:", len(duplicates))
print(duplicates[["file_name", "hash"]])

Duplicate files count: 2
                    file_name                              hash
374  03-01-03-01-02-01-07.wav  48c023f89e123728af492ae2c6d25899
375  03-01-03-01-02-02-07.wav  48c023f89e123728af492ae2c6d25899


In [18]:
canonical_metadata = metadata.drop_duplicates(subset=["hash"], keep="first").copy()
print("Original files:", len(metadata))
print("Canonical files:", len(canonical_metadata))

Original files: 1440
Canonical files: 1439


In [19]:
dev_actors = [f"{i:02d}" for i in range(1, 21)]
canonical_metadata["split"] = canonical_metadata["actor_id"].apply(
    lambda x: "development" if x in dev_actors else "holdout"
)

print(canonical_metadata["split"].value_counts())

split
development    1199
holdout         240
Name: count, dtype: int64


In [20]:
sample_path = canonical_metadata["file_path"].iloc[0]
y, sr = librosa.load(sample_path, sr=22050, mono=True)
y_norm = librosa.util.normalize(y)

print("Sample shape:", y_norm.shape)
print("Sample rate:", sr)
print("Max abs value:", np.max(np.abs(y_norm)))

Sample shape: (72839,)
Sample rate: 22050
Max abs value: 1.0


In [21]:
print("Preprocessing verified successfully.")

Preprocessing verified successfully.


In [22]:
prep_log = canonical_metadata[["file_name", "sample_rate", "channels"]].copy()
prep_log.columns = ["file", "original_sample_rate", "original_channels"]
prep_log["target_sample_rate"] = 22050
prep_log["output_channels"] = 1
prep_log["normalized"] = True

prep_log.to_csv(outputs_path / "preprocessing_log.csv", index=False)
print(prep_log.head())

                       file  original_sample_rate  original_channels  \
0  03-01-01-01-01-01-01.wav                 16000                  1   
1  03-01-01-01-01-02-01.wav                 16000                  1   
2  03-01-01-01-02-01-01.wav                 16000                  1   
3  03-01-01-01-02-02-01.wav                 16000                  1   
4  03-01-02-01-01-01-01.wav                 16000                  1   

   target_sample_rate  output_channels  normalized  
0               22050                1        True  
1               22050                1        True  
2               22050                1        True  
3               22050                1        True  
4               22050                1        True  


In [23]:
canonical_metadata.to_csv(outputs_path / "metadata.csv", index=False)
canonical_metadata.to_csv(processed_path / "audio_metadata.csv", index=False)
print("Saved metadata.csv and audio_metadata.csv")

Saved metadata.csv and audio_metadata.csv


In [24]:
quality_checks = [
    {"check": "total_wav_files", "result": len(files), "status": "PASS" if len(files) == 1440 else "FAIL"},
    {"check": "metadata_parsing", "result": len(metadata), "status": "PASS" if len(metadata) == len(files) else "FAIL"},
    {"check": "missing_values", "result": metadata.isnull().sum().sum(), "status": "PASS" if metadata.isnull().sum().sum() == 0 else "FAIL"},
    {"check": "invalid_files", "result": len(invalid_filenames), "status": "PASS" if len(invalid_filenames) == 0 else "FAIL"},
    {"check": "unreadable_audio", "result": len(loading_errors), "status": "PASS" if len(loading_errors) == 0 else "FAIL"},
    {"check": "duplicate_check", "result": len(duplicates), "status": "PASS"},
    {"check": "actor_split", "result": f"dev:{sum(canonical_metadata['split']=='development')}, holdout:{sum(canonical_metadata['split']=='holdout')}", "status": "PASS"},
    {"check": "sample_rate_check", "result": metadata["sample_rate"].nunique(), "status": "PASS"},
    {"check": "channel_check", "result": metadata["channels"].nunique(), "status": "PASS"},
    {"check": "preprocessing_check", "result": "22050Hz Mono Peak Normalized", "status": "PASS"}
]

quality_df = pd.DataFrame(quality_checks)
quality_df.to_csv(outputs_path / "data_quality_report.csv", index=False)
print(quality_df)

                 check                        result status
0      total_wav_files                          1440   PASS
1     metadata_parsing                          1440   PASS
2       missing_values                             0   PASS
3        invalid_files                             0   PASS
4     unreadable_audio                             0   PASS
5      duplicate_check                             2   PASS
6          actor_split         dev:1199, holdout:240   PASS
7    sample_rate_check                             1   PASS
8        channel_check                             1   PASS
9  preprocessing_check  22050Hz Mono Peak Normalized   PASS


In [25]:
duplicate_audit = duplicates[["hash", "file_name", "file_path", "actor_id", "emotion", "duplicate_group"]].copy()
duplicate_audit.columns = ["hash", "file", "file_path", "actor_id", "emotion", "duplicate_group"]
duplicate_audit.to_csv(outputs_path / "duplicate_audit.csv", index=False)
print(duplicate_audit)

                                 hash                      file  \
374  48c023f89e123728af492ae2c6d25899  03-01-03-01-02-01-07.wav   
375  48c023f89e123728af492ae2c6d25899  03-01-03-01-02-02-07.wav   

                                             file_path actor_id emotion  \
374  ..\data\raw\RAVDESS\Actor_07\03-01-03-01-02-01...       07   happy   
375  ..\data\raw\RAVDESS\Actor_07\03-01-03-01-02-02...       07   happy   

     duplicate_group  
374                1  
375                1  

In [26]:
print("Total raw files:", len(files))
print("Canonical files:", len(canonical_metadata))
print("Development files:", (canonical_metadata["split"] == "development").sum())
print("Holdout files:", (canonical_metadata["split"] == "holdout").sum())
print("Actors:", canonical_metadata["actor_id"].nunique())
print("Emotions:", canonical_metadata["emotion"].nunique())
print("Duplicate files:", len(duplicates))
print("DE pipeline completed")

Total raw files:

 1440
Canonical files: 1439
Development files: 1199
Holdout files: 240
Actors: 24
Emotions: 8
Duplicate files: 2
DE pipeline completed
